In [33]:
from libpysal.weights import Queen
from spreg import OLS
from esda.moran import Moran
from spreg import ML_Lag
import geopandas as gpd

In [34]:
file_path = '/home1/kojoseph/ah-la-paper/data/del_t_hoods.parquet'
del_t_hoods = gpd.read_parquet(file_path)

In [35]:
del_t_hoods.columns

Index(['external_i', 'name', 'location', 'latitude', 'slug_1', 'sqmi',
       'display_na', 'set', 'slug', 'longitude', 'name_1', 'kind', 'type',
       'geometry', 'mean_delta_t2', 'mean_delta_tc', 'ahf_b', 'ahf_m', 'ahf_t',
       'ahf_tot', 'f_traffic', 'f_building', 'f_metabolism'],
      dtype='str')

# 2m air temp

In [36]:
gdf = del_t_hoods.copy()
# Drop NaNs FIRST
gdf = gdf.dropna(subset=['mean_delta_t2', 'ahf_tot'])
w = Queen.from_dataframe(gdf)
w.transform = 'r'   # row-standardize

('WARNING: ', 5, ' is an island (no neighbors)')
('WARNING: ', 44, ' is an island (no neighbors)')
('WARNING: ', 48, ' is an island (no neighbors)')
('WARNING: ', 49, ' is an island (no neighbors)')
('WARNING: ', 59, ' is an island (no neighbors)')
('WARNING: ', 61, ' is an island (no neighbors)')
('WARNING: ', 67, ' is an island (no neighbors)')
('WARNING: ', 69, ' is an island (no neighbors)')
('WARNING: ', 71, ' is an island (no neighbors)')
('WARNING: ', 83, ' is an island (no neighbors)')
('WARNING: ', 113, ' is an island (no neighbors)')
('WARNING: ', 124, ' is an island (no neighbors)')
('WARNING: ', 137, ' is an island (no neighbors)')
('WARNING: ', 146, ' is an island (no neighbors)')
('WARNING: ', 149, ' is an island (no neighbors)')
('WARNING: ', 150, ' is an island (no neighbors)')
('WARNING: ', 153, ' is an island (no neighbors)')
('WARNING: ', 162, ' is an island (no neighbors)')
('WARNING: ', 175, ' is an island (no neighbors)')
('WARNING: ', 180, ' is an island (no neig

/tmp/SLURM_7620164/ipykernel_3745830/2353300016.py:4: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(gdf)
/home1/kojoseph/.conda/envs/spatial-stats/lib/python3.14/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 50 disconnected components.
 There are 38 islands with ids: 5, 44, 48, 49, 59, 61, 67, 69, 71, 83, 113, 124, 137, 146, 149, 150, 153, 162, 175, 180, 186, 187, 199, 203, 207, 220, 221, 223, 231, 236, 240, 243, 246, 248, 250, 259, 260, 261.
  W.__init__(self, neighbors, ids=ids, **kw)


In [37]:
# cheeck for missing values / nans
print(gdf.isna().sum())

# print which neighborhoods have missing values for mean_delta_t2
print(gdf[gdf['mean_delta_t2'].isna()]['name'])

external_i         0
name               0
location           0
latitude           0
slug_1           270
sqmi               0
display_na         0
set                0
slug               0
longitude          0
name_1           270
kind               0
type               0
geometry           0
mean_delta_t2      0
mean_delta_tc      0
ahf_b              0
ahf_m              0
ahf_t              0
ahf_tot            0
f_traffic          0
f_building         0
f_metabolism       0
dtype: int64
Series([], Name: name, dtype: str)


In [38]:
# fit baseline OLS
y = gdf["mean_delta_t2"].values.reshape(-1,1)
X = gdf[["ahf_tot"]].values

ols = OLS(y, X)

In [39]:
# get OLS residuals
residuals = ols.u
len(residuals)

270

In [40]:
moran = Moran(residuals, w)

print("Moran's I:", moran.I)
print("p-value:", moran.p_sim)

Moran's I: 0.5914394856832984
p-value: 0.001


In [41]:
y = gdf["mean_delta_t2"].values.reshape(-1, 1)
X = gdf[["ahf_tot"]].values
slm = ML_Lag(
    y,
    X,
    w=w,
    name_y='mean_delta_t2',
    name_x=['mean_ahf']
)

print(slm.summary)

# save output as file
out_path = '/home1/kojoseph/ah-la-paper/data/t2_slm_summary.txt'
with open(out_path, 'w') as f:
    f.write(slm.summary)

ML_Lag
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :mean_delta_t2                Number of Observations:         270
Mean dependent var  :      0.2082                Number of Variables   :           3
S.D. dependent var  :      0.1285                Degrees of Freedom    :         267
Pseudo R-squared    :      0.6346
Spatial Pseudo R-squared:  0.5053
Log likelihood      :    303.2753
Sigma-square ML     :      0.0060                Akaike info criterion :    -600.551
S.E of regression   :      0.0775                Schwarz criterion     :    -589.755

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------

In [42]:
# save OLS summary 
print(ols.summary)
out_path = '/home1/kojoseph/ah-la-paper/data/t2_ols_summary.txt'
with open(out_path, 'w') as f:
    f.write(ols.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :        None
Dependent Variable  :     dep_var                Number of Observations:         270
Mean dependent var  :      0.2082                Number of Variables   :           2
S.D. dependent var  :      0.1285                Degrees of Freedom    :         268
R-squared           :      0.5341
Adjusted R-squared  :      0.5324
Sum squared residual:     2.06789                F-statistic           :    307.2623
Sigma-square        :       0.008                Prob(F-statistic)     :   2.346e-46
S.E. of regression  :       0.088                Log likelihood        :     274.592
Sigma-square ML     :       0.008                Akaike info criterion :    -545.184
S.E of regression ML:      0.0875                Schwarz criterion     :    -537.987

-----------------

In [43]:
print(gdf[['ahf_tot', 'mean_delta_t2']].describe())
print(gdf[['ahf_tot', 'mean_delta_t2']].corr())

          ahf_tot  mean_delta_t2
count  270.000000     270.000000
mean     8.389874       0.208231
std      6.355105       0.128456
min      0.030077      -0.048646
25%      3.929598       0.107885
50%      7.712531       0.226549
75%     11.610414       0.311716
max     36.269520       0.520016
                ahf_tot  mean_delta_t2
ahf_tot        1.000000       0.730839
mean_delta_t2  0.730839       1.000000


# Canopy Air Temp

In [44]:
gdf = del_t_hoods.copy()
# Drop NaNs FIRST
gdf = gdf.dropna(subset=['mean_delta_tc', 'ahf_tot'])
w = Queen.from_dataframe(gdf)
w.transform = 'r'   # row-standardize

('WARNING: ', 5, ' is an island (no neighbors)')
('WARNING: ', 44, ' is an island (no neighbors)')
('WARNING: ', 48, ' is an island (no neighbors)')
('WARNING: ', 49, ' is an island (no neighbors)')
('WARNING: ', 59, ' is an island (no neighbors)')
('WARNING: ', 61, ' is an island (no neighbors)')
('WARNING: ', 67, ' is an island (no neighbors)')
('WARNING: ', 69, ' is an island (no neighbors)')
('WARNING: ', 71, ' is an island (no neighbors)')
('WARNING: ', 83, ' is an island (no neighbors)')
('WARNING: ', 113, ' is an island (no neighbors)')
('WARNING: ', 124, ' is an island (no neighbors)')
('WARNING: ', 137, ' is an island (no neighbors)')
('WARNING: ', 146, ' is an island (no neighbors)')
('WARNING: ', 149, ' is an island (no neighbors)')
('WARNING: ', 150, ' is an island (no neighbors)')
('WARNING: ', 153, ' is an island (no neighbors)')
('WARNING: ', 162, ' is an island (no neighbors)')
('WARNING: ', 175, ' is an island (no neighbors)')
('WARNING: ', 180, ' is an island (no neig

/tmp/SLURM_7620164/ipykernel_3745830/2427588101.py:4: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(gdf)
/home1/kojoseph/.conda/envs/spatial-stats/lib/python3.14/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 50 disconnected components.
 There are 38 islands with ids: 5, 44, 48, 49, 59, 61, 67, 69, 71, 83, 113, 124, 137, 146, 149, 150, 153, 162, 175, 180, 186, 187, 199, 203, 207, 220, 221, 223, 231, 236, 240, 243, 246, 248, 250, 259, 260, 261.
  W.__init__(self, neighbors, ids=ids, **kw)


In [45]:
# cheeck for missing values / nans
print(gdf.isna().sum())

# print which neighborhoods have missing values for mean_delta_t2
print(gdf[gdf['mean_delta_tc'].isna()]['name'])

external_i         0
name               0
location           0
latitude           0
slug_1           270
sqmi               0
display_na         0
set                0
slug               0
longitude          0
name_1           270
kind               0
type               0
geometry           0
mean_delta_t2      0
mean_delta_tc      0
ahf_b              0
ahf_m              0
ahf_t              0
ahf_tot            0
f_traffic          0
f_building         0
f_metabolism       0
dtype: int64
Series([], Name: name, dtype: str)


In [46]:
# fit baseline OLS
y = gdf["mean_delta_tc"].values.reshape(-1,1)
X = gdf[["ahf_tot"]].values

ols = OLS(y, X)

# get OLS residuals
residuals = ols.u
len(residuals)

moran = Moran(residuals, w)

print("Moran's I:", moran.I)
print("p-value:", moran.p_sim)

Moran's I: 0.32785727670618553
p-value: 0.001


In [47]:
y = gdf["mean_delta_tc"].values.reshape(-1, 1)
X = gdf[["ahf_tot"]].values
slm = ML_Lag(
    y,
    X,
    w=w,
    name_y='mean_delta_tc',
    name_x=['mean_ahf']
)

print(slm.summary)

# save output as file
out_path = '/home1/kojoseph/ah-la-paper/data/tc_slm_summary.txt'
with open(out_path, 'w') as f:
    f.write(slm.summary)

ML_Lag
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :mean_delta_tc                Number of Observations:         270
Mean dependent var  :      0.9616                Number of Variables   :           3
S.D. dependent var  :      0.6577                Degrees of Freedom    :         267
Pseudo R-squared    :      0.7253
Spatial Pseudo R-squared:  0.7174
Log likelihood      :    -95.1269
Sigma-square ML     :      0.1184                Akaike info criterion :     196.254
S.E of regression   :      0.3440                Schwarz criterion     :     207.049

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------

In [48]:
# save OLS summary 
print(ols.summary)
out_path = '/home1/kojoseph/ah-la-paper/data/tc_ols_summary.txt'
with open(out_path, 'w') as f:
    f.write(ols.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :        None
Dependent Variable  :     dep_var                Number of Observations:         270
Mean dependent var  :      0.9616                Number of Variables   :           2
S.D. dependent var  :      0.6577                Degrees of Freedom    :         268
R-squared           :      0.7235
Adjusted R-squared  :      0.7224
Sum squared residual:     32.1769                F-statistic           :    701.1552
Sigma-square        :       0.120                Prob(F-statistic)     :   8.923e-77
S.E. of regression  :       0.347                Log likelihood        :     -95.945
Sigma-square ML     :       0.119                Akaike info criterion :     195.890
S.E of regression ML:      0.3452                Schwarz criterion     :     203.086

-----------------

In [49]:
print(gdf[['ahf_tot', 'mean_delta_tc']].describe())
print(gdf[['ahf_tot', 'mean_delta_tc']].corr())

          ahf_tot  mean_delta_tc
count  270.000000     270.000000
mean     8.389874       0.961579
std      6.355105       0.657696
min      0.030077       0.000000
25%      3.929598       0.587700
50%      7.712531       0.935108
75%     11.610414       1.273519
max     36.269520       4.879354
                ahf_tot  mean_delta_tc
ahf_tot        1.000000       0.850571
mean_delta_tc  0.850571       1.000000
